# LeetCode 958 — Check Completeness of a Binary Tree bằng Python

Link đề: https://leetcode.com/problems/check-completeness-of-a-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree (cây), Breadth-First Search (BFS), Binary Tree (cây nhị phân)

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def isCompleteTree(self, root: Optional[TreeNode]) -> bool:
        ...
```

Ghi chú: **cây nhị phân hoàn chỉnh (complete)** là cây mà mọi tầng, trừ có thể tầng cuối, đều đầy; ở tầng cuối các nút dồn **sát về bên trái** nhất có thể. Đặc trưng quen thuộc: nếu duyệt theo từng tầng (BFS) và **ghi cả các con rỗng**, thì sau **nút rỗng đầu tiên** không còn nút thật nào. Tương đương: đánh số nút theo kiểu **đống** (gốc `1`, con trái `2i`, con phải `2i + 1`) thì chỉ số lớn nhất bằng **số nút**. Bài này **tự dựng `TreeNode`** và hàm dựng/xuất cây theo danh sách kiểu LeetCode để kiểm thử.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: kiểm tra một cây nhị phân có là cây hoàn chỉnh không (các tầng đầy, tầng cuối dồn trái).
2. Nắm các công cụ Python cần thiết: lớp `TreeNode`, hàm dựng cây từ danh sách theo mức (có `None`), `collections.deque` cho BFS, đệ quy trả về nhiều giá trị (bộ ba), và chỉ số đống (`2i`, `2i + 1`).
3. Hiểu nguyên lý: cây hoàn chỉnh ⇔ **không có nút thật nào xuất hiện sau một nút rỗng trong duyệt theo tầng** ⇔ **chỉ số đống lớn nhất bằng số nút** ⇔ cấu trúc trùng với cây hoàn chỉnh chuẩn `n` nút.
4. Đi qua 4 cách: so với cây hoàn chỉnh chuẩn `n` nút (brute, oracle theo định nghĩa), BFS với nút rỗng (cách nộp chính), chỉ số đống, và đệ quy theo bộ ba (hoàn chỉnh, hoàn hảo, chiều cao); đối chiếu bằng cây sinh có chủ đích.
5. Tránh các lỗi kinh điển: nhầm với cây **đầy (full)**, quên kiểm tra các nút còn lại sau nút rỗng, chỉ kiểm tra "có con phải thì có con trái", và đánh số chỉ số 0-based sai.

## 2. Đề bài & ví dụ

Cho nút gốc `root` của một cây nhị phân, xác định xem nó có phải **cây nhị phân hoàn chỉnh** hay không.

Trong cây hoàn chỉnh, mọi tầng, ngoại trừ có thể tầng cuối cùng, đều được **lấp đầy**, và mọi nút ở tầng cuối **dồn sát về bên trái** nhất có thể. Tầng cuối `h` có từ `1` đến `2^h` nút (tính cả hai đầu).

**Ví dụ 1:** `root = [1,2,3,4,5,6]` -> `true` (hai tầng đầu đầy, tầng cuối `{4,5,6}` sát trái).

**Ví dụ 2:** `root = [1,2,3,4,5,null,7]` -> `false` (nút `7` không sát trái nhất có thể).

**Ràng buộc:**

- Số nút trong cây nằm trong `[1, 100]`.
- `1 <= Node.val <= 1000`

In [1]:
from typing import List, Optional
from collections import deque
import random, time

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(vals: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ danh sách theo mức kiểu LeetCode (None là nút trống; chỉ nút thật có hai phần tử con trong danh sách)."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = deque([root]), 1
    while queue and i < len(vals):
        node = queue.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root

def tree_to_list(root: Optional[TreeNode]) -> List[Optional[int]]:
    """Xuất cây theo mức kiểu LeetCode (bỏ None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def count_nodes(root: Optional[TreeNode]) -> int:
    return 0 if root is None else 1 + count_nodes(root.left) + count_nodes(root.right)

def shape(root: Optional[TreeNode]):
    return None if root is None else (shape(root.left), shape(root.right))

examples = [
    ([1, 2, 3, 4, 5, 6], True),
    ([1, 2, 3, 4, 5, None, 7], False),
]
for vals, expected in examples:
    t = build_tree(vals)
    print(f"root={vals} -> số nút {count_nodes(t)}, danh sách theo mức {tree_to_list(t)}, kỳ vọng {expected}")
    assert tree_to_list(t) == vals

root=[1, 2, 3, 4, 5, 6] -> số nút 6, danh sách theo mức [1, 2, 3, 4, 5, 6], kỳ vọng True
root=[1, 2, 3, 4, 5, None, 7] -> số nút 6, danh sách theo mức [1, 2, 3, 4, 5, None, 7], kỳ vọng False


## 3. Kiến thức nền

### 3.1. Duyệt theo tầng (BFS) và nút rỗng

BFS bằng `deque`: lấy nút ở đầu, đẩy hai con (kể cả `None`) vào cuối. Nếu **ghi lại cả `None`**, thứ tự duyệt cho ta "dãy theo mức" của cây (cây hoàn chỉnh có dãy này **không có nút thật sau nút `None`**).

In [2]:
def level_sequence(root):
    """Dãy duyệt theo tầng gồm cả None, dừng sau khi đã xử lý mọi nút thật."""
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    return out

for vals, _ in examples:
    print(level_sequence(build_tree(vals)))

[1, 2, 3, 4, 5, 6, None, None, None, None, None, None, None]
[1, 2, 3, 4, 5, None, 7, None, None, None, None, None, None]


### 3.2. Chỉ số kiểu đống

Gán gốc chỉ số `1`; nút chỉ số `i` có con trái `2i` và con phải `2i + 1`. Cây hoàn chỉnh `n` nút chiếm **đúng** các chỉ số `1..n` (không thủng): ngược lại một chỉ số lớn hơn `n` xuất hiện khi có "lỗ".

In [3]:
def indices(root):
    res = {}
    def go(node, i):
        if node is None:
            return
        res[i] = node.val
        go(node.left, 2 * i)
        go(node.right, 2 * i + 1)
    go(root, 1)
    return res

for vals, _ in examples:
    t = build_tree(vals)
    idx = indices(t)
    print(f"{vals}: chỉ số {sorted(idx)} | số nút {count_nodes(t)} | chỉ số lớn nhất {max(idx)}")

[1, 2, 3, 4, 5, 6]: chỉ số [1, 2, 3, 4, 5, 6] | số nút 6 | chỉ số lớn nhất 6
[1, 2, 3, 4, 5, None, 7]: chỉ số [1, 2, 3, 4, 5, 7] | số nút 6 | chỉ số lớn nhất 7


### 3.3. Cây đầy và cây hoàn chỉnh khác nhau

**Cây đầy (full):** mỗi nút có `0` hoặc `2` con. **Cây hoàn chỉnh:** các tầng đầy trừ tầng cuối dồn trái. Một cây hoàn chỉnh có thể có nút chỉ một con (con trái duy nhất ở cuối); một cây đầy có thể không hoàn chỉnh.

In [4]:
only_left = build_tree([1, 2, 3, 4])                     # hoàn chỉnh nhưng không đầy (nút 2 chỉ có con trái)
full_not_complete = build_tree([1, 2, 3, None, None, 4, 5])  # đầy nhưng không hoàn chỉnh
print(tree_to_list(only_left), "| tree_to_list full_not_complete:", tree_to_list(full_not_complete))

[1, 2, 3, 4] | tree_to_list full_not_complete: [1, 2, 3, None, None, 4, 5]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `[1,2,3,4,5,null,7]`. Duyệt theo tầng, ghi cả con rỗng: `1, 2, 3, 4, 5, None (con trái của 3), 7, ...`. Sau `None` có nút thật `7` ⇒ **không hoàn chỉnh**.

Ví dụ 1: `[1,2,3,4,5,6]`: dãy `1, 2, 3, 4, 5, 6, None, None, None, None, None, None, None`: sau `None` đầu tiên chỉ còn `None` ⇒ hoàn chỉnh.

Cách nhìn đống: ví dụ 2 có nút `7` ở chỉ số `7` nhưng chỉ có `6` nút: chỉ số lớn nhất `7 != 6` ⇒ không hoàn chỉnh.

## 5. Cách 1 — Brute Force: so với cây hoàn chỉnh chuẩn `n` nút

**Ý tưởng.** Đếm số nút `n`; dựng **hình dạng cây hoàn chỉnh chuẩn** `n` nút theo định nghĩa: hình dạng của nút chỉ số `i` là `None` nếu `i > n`, ngược lại `(hình(2i), hình(2i + 1))`. Cây là hoàn chỉnh ⇔ hình dạng của nó **bằng** hình dạng chuẩn. Đây là định nghĩa trực tiếp (dựa trên cấu trúc đống) nên cũng là **oracle**, hoàn toàn khác BFS.

In [5]:
def complete_shape(n: int, i: int = 1):
    if i > n:
        return None
    return (complete_shape(n, 2 * i), complete_shape(n, 2 * i + 1))

def is_complete_shape(root: Optional[TreeNode]) -> bool:
    return shape(root) == complete_shape(count_nodes(root))

for vals, expected in examples:
    got = is_complete_shape(build_tree(vals))
    print(vals, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3, 4, 5, 6] -> True OK
[1, 2, 3, 4, 5, None, 7] -> False OK


**Độ phức tạp và nút thắt.** `O(n)` để đếm và dựng hình dạng chuẩn và so sánh, nhưng cần dựng **thêm** một cấu trúc cỡ `n`. Có cách chỉ cần **một lượt duyệt** mà không dựng cây phụ.

## 6. Ý tưởng thuật toán / bước đột phá

Cây hoàn chỉnh có các nút thật chiếm một **tiền tố** của thứ tự theo tầng (trái sang phải, trên xuống dưới) nếu ta tính cả vị trí trống. Do đó:

- **BFS có nút rỗng:** sau khi gặp một `None`, mọi phần tử sau cũng phải là `None`.
- **Chỉ số đống:** các nút chiếm đúng `1..n` ⇔ chỉ số lớn nhất `= n`.
- **Đệ quy:** kết hợp từ hai cây con ba thông tin (hoàn chỉnh, hoàn hảo, chiều cao).

| Cách | Ý tưởng |
|---|---|
| 2 | BFS: gặp `None` thì mọi phần tử còn lại phải là `None` (nộp) |
| 3 | Chỉ số đống lớn nhất bằng số nút |
| 4 | Đệ quy trả `(hoàn chỉnh, hoàn hảo, chiều cao)` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — BFS và nút rỗng (cách nộp chính)

**Giải thích từng đoạn.**

- `queue = deque([root])`, `seen_none = False`.
- Lấy `node`: nếu `node is None` thì `seen_none = True`; ngược lại nếu `seen_none` (đã có `None` trước đó) thì trả `False`; nếu không thì đẩy `node.left`, `node.right` (kể cả `None`) vào hàng đợi.
- Hết hàng đợi: `True`.

In [6]:
def complete_bfs(root: Optional[TreeNode]) -> bool:
    queue = deque([root])
    seen_none = False
    while queue:
        node = queue.popleft()
        if node is None:
            seen_none = True
            continue
        if seen_none:
            return False
        queue.append(node.left)
        queue.append(node.right)
    return True

for vals, expected in examples:
    got = complete_bfs(build_tree(vals))
    print(vals, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3, 4, 5, 6] -> True OK
[1, 2, 3, 4, 5, None, 7] -> False OK


**Code nộp LeetCode (Cách 2):**

In [7]:
class Solution:
    def isCompleteTree(self, root: Optional[TreeNode]) -> bool:
        from collections import deque
        queue = deque([root])
        seen_none = False
        while queue:
            node = queue.popleft()
            if node is None:
                seen_none = True
                continue
            if seen_none:
                return False
            queue.append(node.left)
            queue.append(node.right)
        return True

sol_bfs = Solution().isCompleteTree
assert [sol_bfs(build_tree(v)) for v, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n)` thời gian (mỗi nút và mỗi con rỗng một lần), `O(n)` bộ nhớ (hàng đợi tối đa một tầng, cỡ `n / 2`).

---

### 7.2. Cách 3 — Chỉ số kiểu đống

**Ý tưởng khác.** Duyệt (BFS hoặc DFS) gán chỉ số `i` cho từng nút (gốc `1`, trái `2i`, phải `2i + 1`), đếm số nút `n` và tìm chỉ số lớn nhất. Hoàn chỉnh ⇔ `max_index == n`.

**Giải thích từng đoạn.** Các chỉ số của các nút khác nhau là **khác nhau** (đường đi từ gốc xác định duy nhất chỉ số). Nếu có `n` nút với chỉ số đôi một khác nhau, tất cả `<= n` thì chúng là đúng `1..n`; ngược lại có chỉ số `> n` (lỗ).

In [8]:
def complete_index(root: Optional[TreeNode]) -> bool:
    count, max_index = 0, 0
    queue = deque([(root, 1)])
    while queue:
        node, i = queue.popleft()
        if node is None:
            continue
        count += 1
        max_index = max(max_index, i)
        queue.append((node.left, 2 * i))
        queue.append((node.right, 2 * i + 1))
    return max_index == count

for vals, expected in examples:
    got = complete_index(build_tree(vals))
    print(vals, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3, 4, 5, 6] -> True OK
[1, 2, 3, 4, 5, None, 7] -> False OK


**Code nộp LeetCode (Cách 3):**

In [9]:
class Solution:
    def isCompleteTree(self, root: Optional[TreeNode]) -> bool:
        from collections import deque
        count, max_index = 0, 0
        queue = deque([(root, 1)])
        while queue:
            node, i = queue.popleft()
            if node is None:
                continue
            count += 1
            max_index = max(max_index, i)
            queue.append((node.left, 2 * i))
            queue.append((node.right, 2 * i + 1))
        return max_index == count

sol_index = Solution().isCompleteTree
assert [sol_index(build_tree(v)) for v, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n)` thời gian, `O(n)` bộ nhớ. Chỉ số có thể lớn (tới `2^n` với cây lệch); Python có số nguyên lớn tùy ý nên không tràn (ở ngôn ngữ khác cần chú ý).

---

### 7.3. Cách 4 — Đệ quy trả bộ ba `(hoàn chỉnh, hoàn hảo, chiều cao)`

**Ý tưởng khác.** Cây **hoàn hảo (perfect)** là cây có mọi tầng đầy. Với một nút, cây con trái/phải có thông tin `(lc, lp, lh)`, `(rc, rp, rh)`:

- hoàn hảo: `lp and rp and lh == rh`;
- hoàn chỉnh: (a) `lp and rc and lh == rh` (trái hoàn hảo, phải hoàn chỉnh cùng chiều cao: tầng cuối nằm ở cây con phải), hoặc (b) `lc and rp and lh == rh + 1` (trái hoàn chỉnh cao hơn một, phải hoàn hảo: tầng cuối nằm ở cây con trái).
- chiều cao: `1 + max(lh, rh)`. Cây rỗng: `(True, True, 0)`.

**Giải thích từng đoạn.** Với (a) cả hai cây con có cùng chiều cao `h` và tầng cuối của cây lớn nằm ở bên phải (trái đầy). Với (b) tầng cuối nằm ở bên trái (phải đầy và thấp hơn một tầng). Trường hợp cả hai hoàn hảo cùng chiều cao thuộc (a).

In [10]:
def complete_recursive(root: Optional[TreeNode]) -> bool:
    def f(node):
        if node is None:
            return True, True, 0
        lc, lp, lh = f(node.left)
        rc, rp, rh = f(node.right)
        perfect = lp and rp and lh == rh
        complete = (lp and rc and lh == rh) or (lc and rp and lh == rh + 1)
        return complete, perfect, 1 + max(lh, rh)

    return f(root)[0]

for vals, expected in examples:
    got = complete_recursive(build_tree(vals))
    print(vals, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3, 4, 5, 6] -> True OK
[1, 2, 3, 4, 5, None, 7] -> False OK


**Code nộp LeetCode (Cách 4):**

In [11]:
class Solution:
    def isCompleteTree(self, root: Optional[TreeNode]) -> bool:
        def f(node):
            if node is None:
                return True, True, 0
            lc, lp, lh = f(node.left)
            rc, rp, rh = f(node.right)
            perfect = lp and rp and lh == rh
            complete = (lp and rc and lh == rh) or (lc and rp and lh == rh + 1)
            return complete, perfect, 1 + max(lh, rh)

        return f(root)[0]

sol_rec = Solution().isCompleteTree
assert [sol_rec(build_tree(v)) for v, _ in examples] == [e for _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n)` thời gian, `O(h)` bộ nhớ ngăn xếp đệ quy (`h` là chiều cao, tới `n` với cây lệch; ràng buộc `n <= 100` nên an toàn).

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 2: `[1,2,3,4,5,null,7]`

In [12]:
def trace_bfs(vals):
    queue = deque([build_tree(vals)])
    seen_none = False
    step = 0
    while queue:
        node = queue.popleft()
        step += 1
        if node is None:
            seen_none = True
            print(f"bước {step}: None (đã thấy None = True)")
            continue
        if seen_none:
            print(f"bước {step}: nút {node.val} xuất hiện SAU một None -> False")
            return False
        print(f"bước {step}: nút {node.val}; đẩy con trái {node.left.val if node.left else None}, con phải {node.right.val if node.right else None}")
        queue.append(node.left)
        queue.append(node.right)
    return True

assert trace_bfs([1, 2, 3, 4, 5, None, 7]) is False
print()
assert trace_bfs([1, 2, 3, 4, 5, 6]) is True

bước 1: nút 1; đẩy con trái 2, con phải 3
bước 2: nút 2; đẩy con trái 4, con phải 5
bước 3: nút 3; đẩy con trái None, con phải 7
bước 4: nút 4; đẩy con trái None, con phải None
bước 5: nút 5; đẩy con trái None, con phải None
bước 6: None (đã thấy None = True)
bước 7: nút 7 xuất hiện SAU một None -> False

bước 1: nút 1; đẩy con trái 2, con phải 3
bước 2: nút 2; đẩy con trái 4, con phải 5
bước 3: nút 3; đẩy con trái 6, con phải None
bước 4: nút 4; đẩy con trái None, con phải None
bước 5: nút 5; đẩy con trái None, con phải None
bước 6: nút 6; đẩy con trái None, con phải None
bước 7: None (đã thấy None = True)
bước 8: None (đã thấy None = True)
bước 9: None (đã thấy None = True)
bước 10: None (đã thấy None = True)
bước 11: None (đã thấy None = True)
bước 12: None (đã thấy None = True)
bước 13: None (đã thấy None = True)


### 8.2. Cách 3 cho ví dụ 2: chỉ số đống

`1 -> 2, 3`; `2 -> 4, 5`; `3 -> (6 trống), 7`. Nút `7` có chỉ số `7` mà chỉ có `6` nút.

In [13]:
t = build_tree([1, 2, 3, 4, 5, None, 7])
print(indices(t), "| số nút", count_nodes(t))
assert max(indices(t)) == 7 and count_nodes(t) == 6 and complete_index(t) is False

{1: 1, 2: 2, 4: 4, 5: 5, 3: 3, 7: 7} | số nút 6


### 8.3. Cách 4: bộ ba tại từng nút của ví dụ 1

Lá: `(True, True, 1)`. Nút `2` (con `4`, `5` cùng lá): trái `(T,T,1)`, phải `(T,T,1)`: hoàn hảo, chiều cao `2`. Nút `3` (con `6` và `None`): trái `(T,T,1)`, phải `(T,T,0)`: hoàn hảo? `lh == rh`? `1 != 0` không; hoàn chỉnh: `lc and rp and lh == rh + 1` ✓. Gốc: trái hoàn hảo `h = 2`, phải hoàn chỉnh `h = 2` ⇒ hoàn chỉnh.

In [14]:
def triple(node):
    if node is None:
        return True, True, 0
    lc, lp, lh = triple(node.left)
    rc, rp, rh = triple(node.right)
    perfect = lp and rp and lh == rh
    complete = (lp and rc and lh == rh) or (lc and rp and lh == rh + 1)
    print(f"nút {node.val}: trái ({lc},{lp},{lh}), phải ({rc},{rp},{rh}) -> hoàn chỉnh={complete}, hoàn hảo={perfect}, cao={1 + max(lh, rh)}")
    return complete, perfect, 1 + max(lh, rh)

assert triple(build_tree([1, 2, 3, 4, 5, 6]))[0] is True

nút 4: trái (True,True,0), phải (True,True,0) -> hoàn chỉnh=True, hoàn hảo=True, cao=1
nút 5: trái (True,True,0), phải (True,True,0) -> hoàn chỉnh=True, hoàn hảo=True, cao=1
nút 2: trái (True,True,1), phải (True,True,1) -> hoàn chỉnh=True, hoàn hảo=True, cao=2
nút 6: trái (True,True,0), phải (True,True,0) -> hoàn chỉnh=True, hoàn hảo=True, cao=1
nút 3: trái (True,True,1), phải (True,True,0) -> hoàn chỉnh=True, hoàn hảo=False, cao=2
nút 1: trái (True,True,2), phải (True,False,2) -> hoàn chỉnh=True, hoàn hảo=False, cao=3


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (BFS có nút rỗng): ngắn, trực quan, `O(n)`.

```python
class Solution:
    def isCompleteTree(self, root: Optional[TreeNode]) -> bool:
        from collections import deque
        queue = deque([root])
        seen_none = False
        while queue:
            node = queue.popleft()
            if node is None:
                seen_none = True
                continue
            if seen_none:
                return False
            queue.append(node.left)
            queue.append(node.right)
        return True
```

In [15]:
class Solution:
    def isCompleteTree(self, root: Optional[TreeNode]) -> bool:
        from collections import deque
        queue = deque([root])
        seen_none = False
        while queue:
            node = queue.popleft()
            if node is None:
                seen_none = True
                continue
            if seen_none:
                return False
            queue.append(node.left)
            queue.append(node.right)
        return True

sol = Solution()
for vals, expected in examples:
    got = sol.isCompleteTree(build_tree(vals))
    print(vals, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3, 4, 5, 6] -> True OK
[1, 2, 3, 4, 5, None, 7] -> False OK


## 10. Vì sao đúng

**(a) Cây hoàn chỉnh ⇔ các nút chiếm các chỉ số đống `1..n`.** Cây hoàn chỉnh `n` nút: mọi tầng trừ tầng cuối đầy và tầng cuối dồn trái, nên nút ở vị trí thứ `k` theo thứ tự tầng có chỉ số đống `k`; vậy các chỉ số là `1..n`. Ngược lại, nếu các `n` nút có chỉ số đống đôi một khác nhau và đều `<= n` thì chúng là chính `1..n`; mỗi chỉ số `k > 1` có cha là chỉ số `k // 2` (tồn tại vì mọi chỉ số `< k` có mặt), nên cây đúng bằng cây hoàn chỉnh chuẩn `n` nút (Cách 1 và 3).

**(b) BFS (Cách 2).** BFS với con rỗng liệt kê các vị trí theo thứ tự tầng và trái sang phải, tức theo chỉ số đống `1, 2, 3, ...` (vị trí `i` là nút thật hoặc `None`; ta chỉ sinh vị trí của các con của nút thật). Cây hoàn chỉnh có các vị trí thật là tiền tố `1..n` của thứ tự này nên mọi `None` đứng sau mọi nút thật. Ngược lại, nếu sau mọi `None` không có nút thật thì tập chỉ số thật là một tiền tố liên tục `1..n` của thứ tự đó ⇒ hoàn chỉnh theo (a).

**(c) Đệ quy (Cách 4).** Cây hoàn chỉnh `T` chiều cao `H >= 1`. Đường trái nhất đi tới tầng cuối nên cây con trái `L` có chiều cao `H - 1`; cây con phải `R` có chiều cao `H - 1` hoặc `H - 2`.
   - Nếu `R` cao `H - 1`: tầng cuối có nút ở `R` nên theo tính "dồn trái" mọi vị trí tầng cuối thuộc `L` đều có nút, tức `L` **hoàn hảo** (cao `H - 1`), còn `R` là cây hoàn chỉnh cao `H - 1`: điều kiện `lp and rc and lh == rh`.
   - Nếu `R` cao `H - 2`: tầng cuối không có nút ở `R`, nên các tầng trên của `R` phải đầy: `R` **hoàn hảo** cao `H - 2`; `L` là cây hoàn chỉnh cao `H - 1`: điều kiện `lc and rp and lh == rh + 1`.
   Ngược lại hai tổ hợp trên đều tạo ra cây hoàn chỉnh. Cây hoàn hảo có hai cây con hoàn hảo cùng chiều cao.

**Kiểm chứng bằng máy.** Oracle **độc lập** với BFS: so hình dạng cây với **cây hoàn chỉnh chuẩn `n` nút** (Cách 1), dựng từ công thức đống (không duyệt theo tầng). Dữ liệu: (1) **mọi hình dạng cây** tới `n = 8` nút (liệt kê hết), (2) cây hoàn chỉnh `n` nút từ `1` đến `100` (chắc chắn `True`), (3) cây hoàn chỉnh bị **đổi một vị trí** (xóa/thêm nút) (chắc chắn đa số `False`), (4) cây ngẫu nhiên.

In [16]:
def all_shapes(n):
    """Mọi hình dạng cây nhị phân có n nút (dạng tuple lồng nhau)."""
    if n == 0:
        return [None]
    out = []
    for k in range(n):
        for l in all_shapes(k):
            for r in all_shapes(n - 1 - k):
                out.append((l, r))
    return out

def tree_from_shape(sh, counter=None):
    if sh is None:
        return None
    if counter is None:
        counter = [0]
    counter[0] += 1
    node = TreeNode(counter[0])
    node.left = tree_from_shape(sh[0], counter)
    node.right = tree_from_shape(sh[1], counter)
    return node

total = 0
complete_count = 0
for n in range(1, 9):
    for sh in all_shapes(n):
        t = tree_from_shape(sh)
        ref = is_complete_shape(t)
        assert complete_bfs(t) == complete_index(t) == complete_recursive(t) == ref, (n, sh)
        total += 1
        complete_count += ref
print(f"mọi hình dạng cây tới n = 8 nút: {total} cây, trong đó {complete_count} hoàn chỉnh (đúng 1 cây cho mỗi n = 1..8)")
assert complete_count == 8

mọi hình dạng cây tới n = 8 nút: 2055 cây, trong đó 8 hoàn chỉnh (đúng 1 cây cho mỗi n = 1..8)


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. So với cây chuẩn | `O(n)` | `O(n)` |
| 2. BFS + nút rỗng | `O(n)` | `O(n)` |
| 3. Chỉ số đống | `O(n)` | `O(n)` |
| 4. Đệ quy bộ ba | `O(n)` | `O(h)` |

Đo thử với `n = 100` (cỡ tối đa), cây hoàn chỉnh và cây gần hoàn chỉnh:

In [17]:
def complete_tree(n):
    """Cây hoàn chỉnh n nút (giá trị 1..n theo thứ tự tầng)."""
    nodes = [None] + [TreeNode(i) for i in range(1, n + 1)]
    for i in range(1, n + 1):
        if 2 * i <= n:
            nodes[i].left = nodes[2 * i]
        if 2 * i + 1 <= n:
            nodes[i].right = nodes[2 * i + 1]
    return nodes[1]

for label, t in (("hoàn chỉnh 100", complete_tree(100)), ("hoàn chỉnh 63", complete_tree(63))):
    row = []
    for name, fn in (("Cách 2", sol_bfs), ("Cách 3", sol_index), ("Cách 4", sol_rec), ("nộp", Solution().isCompleteTree)):
        t0 = time.perf_counter()
        for _ in range(2000):
            r = fn(t)
        row.append(f"{name}: {1000 * (time.perf_counter() - t0) / 2000:7.3f} ms")
        assert r is True
    print(f"{label:16} " + " | ".join(row))

hoàn chỉnh 100   Cách 2:   0.010 ms | Cách 3:   0.017 ms | Cách 4:   0.021 ms | nộp:   0.011 ms


hoàn chỉnh 63    Cách 2:   0.007 ms | Cách 3:   0.011 ms | Cách 4:   0.015 ms | nộp:   0.008 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle (cây hoàn chỉnh chuẩn), liệt kê **mọi hình dạng** tới `n = 8`, cây sinh có chủ đích (hoàn chỉnh chắc chắn `True`) và cây ngẫu nhiên đúng ràng buộc (`1 <= n <= 100`).

In [18]:
def solutions_list():
    return {
        "bfs": complete_bfs,
        "index": complete_index,
        "recursive": complete_recursive,
        "sol_bfs": sol_bfs,
        "sol_index": sol_index,
        "sol_rec": sol_rec,
        "final": Solution().isCompleteTree,
    }

def run_all(root, expected=None):
    ref = is_complete_shape(root)
    for name, fn in solutions_list().items():
        assert fn(root) == ref, (name, tree_to_list(root))
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", tree_to_list(root), expected, ref)
    return ref

# 12.1 ví dụ đề
for vals, expected in examples:
    assert run_all(build_tree(vals), expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1], True),                                    # một nút
    ([1, 2], True),                                 # con trái duy nhất
    ([1, None, 2], False),                          # con phải mà không có con trái
    ([1, 2, 3], True),                              # đầy hai tầng
    ([1, 2, 3, 4], True),                           # tầng cuối dồn trái
    ([1, 2, 3, None, 4], False),                    # tầng cuối có lỗ ở bên trái
    ([1, 2, 3, 4, 5, 6, 7], True),                  # cây hoàn hảo 3 tầng
    ([1, 2, 3, 4, 5, 6, 7, 8], True),
    ([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], True),
    ([1, 2, 3, 4, 5, None, 7], False),              # ví dụ 2
    ([1, 2, 3, None, None, 4, 5], False),           # cây đầy nhưng không hoàn chỉnh
    ([1, 2, 3, 4, None, 5], False),                 # 5 là con trái của 3 nhưng 2 chỉ có con trái 4 rồi... (tầng cuối: 4, _, 5)
    ([1, 2, None, 3], False),                       # tầng 2 thiếu con phải nhưng tầng 3 có nút
    ([1, 2, 3, 4, 5, 6, None, 7], False),           # tầng 4 xuất hiện khi tầng 3 chưa đầy
]
for vals, expected in edge:
    assert run_all(build_tree(vals), expected) == expected, vals
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca


In [19]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm trên mọi hình dạng nhỏ, xác nhận lời giải đúng không bị lỗi)
def wrong_full_tree(root):                           # nhầm với cây đầy: mỗi nút 0 hoặc 2 con
    if root is None:
        return True
    if (root.left is None) != (root.right is None):
        return False
    return wrong_full_tree(root.left) and wrong_full_tree(root.right)

def wrong_bfs_no_rest_check(root):                   # BFS: gặp None đầu tiên thì dừng và trả True (không kiểm tra phần còn lại)
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            return True
        queue.append(node.left)
        queue.append(node.right)
    return True

def wrong_right_needs_left(root):                    # chỉ kiểm tra "có con phải thì có con trái"
    if root is None:
        return True
    if root.right is not None and root.left is None:
        return False
    return wrong_right_needs_left(root.left) and wrong_right_needs_left(root.right)

def wrong_index_zero_based(root):                    # chỉ số 0-based với con 2i và 2i + 1 (gốc 0: cả hai con của gốc dùng chỉ số 0 và 1)
    count, max_index = 0, 0
    queue = deque([(root, 0)])
    while queue:
        node, i = queue.popleft()
        if node is None:
            continue
        count += 1
        max_index = max(max_index, i)
        queue.append((node.left, 2 * i))
        queue.append((node.right, 2 * i + 1))
    return max_index == count - 1

def find_counter(wrong):
    for n in range(1, 8):
        for sh in all_shapes(n):
            t = tree_from_shape(sh)
            if wrong(t) != is_complete_shape(t):
                return t
    return None

counters = {}
for name, fn in (("nhầm với cây đầy", wrong_full_tree), ("BFS không kiểm tra phần còn lại", wrong_bfs_no_rest_check),
                 ("chỉ kiểm tra con phải cần con trái", wrong_right_needs_left), ("chỉ số 0-based sai", wrong_index_zero_based)):
    t = find_counter(fn)
    assert t is not None, name
    counters[name] = t
    print(f"{name:36} cây {tree_to_list(t)}: sai -> {fn(t)}, đúng -> {is_complete_shape(t)}")
    run_all(t)
print("12.3 phản ví dụ: OK")

nhầm với cây đầy                     cây [1, 2]: sai -> False, đúng -> True
BFS không kiểm tra phần còn lại      cây [1, None, 2]: sai -> True, đúng -> False
chỉ kiểm tra con phải cần con trái   cây [1, 2, None, 3]: sai -> True, đúng -> False
chỉ số 0-based sai                   cây [1, None, 2]: sai -> True, đúng -> False
12.3 phản ví dụ: OK


In [20]:
# 12.4 cây sinh có chủ đích: cây hoàn chỉnh n nút (n = 1..100) luôn True; mỗi cây được "phá" bằng xóa lá / dời nút
for n in range(1, 101):
    t = complete_tree(n)
    assert run_all(t) is True, n
    assert count_nodes(t) == n

def collect_nodes(root):
    out, stack = [], [root]
    while stack:
        x = stack.pop()
        if x:
            out.append(x)
            stack.append(x.left)
            stack.append(x.right)
    return out

rng = random.Random(4958)
false_count = 0
for _ in range(600):
    n = rng.randint(2, 100)
    t = complete_tree(n)
    # phá: cắt một cây con ngẫu nhiên (nút trong) rồi gắn nó vào chỗ trống khác, hoặc xóa một lá giữa
    nodes = collect_nodes(t)
    victim = rng.choice(nodes[1:]) if len(nodes) > 1 else None
    if victim is None:
        continue
    for x in nodes:
        if x.left is victim:
            x.left = None
        if x.right is victim:
            x.right = None
    ref = run_all(t)                                 # có thể True (nếu xóa lá cuối) hoặc False
    false_count += not ref
print(f"12.4 cây hoàn chỉnh n = 1..100 và 600 cây bị phá: OK ({false_count} cây không hoàn chỉnh)")

# 12.5 cây ngẫu nhiên đúng ràng buộc (1..100 nút, giá trị 1..1000)
def random_tree(rng, n):
    nodes = [TreeNode(rng.randint(1, 1000))]
    slots = [(nodes[0], "left"), (nodes[0], "right")]
    for _ in range(n - 1):
        idx = rng.randrange(len(slots))
        slots[idx], slots[-1] = slots[-1], slots[idx]
        parent, side = slots.pop()
        node = TreeNode(rng.randint(1, 1000))
        setattr(parent, side, node)
        slots += [(node, "left"), (node, "right")]
    return nodes[0]

rng = random.Random(5958)
trues = 0
for _ in range(1500):
    n = rng.randint(1, 100)
    t = random_tree(rng, n)
    assert count_nodes(t) == n
    trues += run_all(t)
print(f"12.5 cây ngẫu nhiên 1..100 nút (1500 cây): OK ({trues} hoàn chỉnh)")

# 12.6 dạng đặc biệt: chuỗi trái dài, chuỗi phải dài, cây hoàn chỉnh + một nút thừa
chain_left = TreeNode(1)
cur = chain_left
for v in range(2, 40):
    cur.left = TreeNode(v)
    cur = cur.left
chain_right = TreeNode(1)
cur = chain_right
for v in range(2, 40):
    cur.right = TreeNode(v)
    cur = cur.right
assert run_all(chain_left) is False and run_all(chain_right) is False
t = complete_tree(31)
leaf = t
while leaf.left:
    leaf = leaf.left
leaf.left = TreeNode(99)                              # thêm tầng mới ở nhánh trái nhất: vẫn hoàn chỉnh (32 nút)
assert run_all(t) is True
t2 = complete_tree(31)
leaf2 = t2
while leaf2.right:
    leaf2 = leaf2.right
leaf2.left = TreeNode(99)                             # thêm tầng mới ở nhánh phải nhất: không hoàn chỉnh
assert run_all(t2) is False
print("12.6 dạng đặc biệt: OK")

# 12.7 tính chất: kết quả không phụ thuộc giá trị nút; thêm nút vào vị trí kế tiếp của cây hoàn chỉnh giữ tính hoàn chỉnh
rng = random.Random(7958)
for _ in range(200):
    n = rng.randint(1, 99)
    t = complete_tree(n)
    for x in collect_nodes(t):
        x.val = rng.randint(1, 1000)
    assert run_all(t) is True
    bigger = complete_tree(n + 1)
    assert run_all(bigger) is True
print("12.7 tính chất (không phụ thuộc giá trị, thêm nút kế tiếp): OK")

12.4 cây hoàn chỉnh n = 1..100 và 600 cây bị phá: OK (576 cây không hoàn chỉnh)


12.5 cây ngẫu nhiên 1..100 nút (1500 cây): OK (37 hoàn chỉnh)
12.6 dạng đặc biệt: OK


12.7 tính chất (không phụ thuộc giá trị, thêm nút kế tiếp): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nhầm với cây đầy (mỗi nút 0 hoặc 2 con)

Cây hoàn chỉnh **không** cần mỗi nút có `0` hoặc `2` con: nút cuối ở tầng áp chót có thể chỉ có con trái.

In [21]:
t = counters["nhầm với cây đầy"]
print("cây", tree_to_list(t), "| sai ->", wrong_full_tree(t), "| đúng ->", Solution().isCompleteTree(t))
assert wrong_full_tree(t) != Solution().isCompleteTree(t)
print("ví dụ [1,2,3,4,5,6]:", wrong_full_tree(build_tree([1, 2, 3, 4, 5, 6])), "(sai) so với True")

cây [1, 2] | sai -> False | đúng -> True
ví dụ [1,2,3,4,5,6]: False (sai) so với True


Vì sao sai: nút `3` của `[1,2,3,4,5,6]` chỉ có con trái `6` nhưng cây vẫn hoàn chỉnh. **Đúng:** dùng thứ tự theo tầng và vị trí trống.

### Lỗi 2 — BFS dừng ở `None` đầu tiên mà không kiểm tra phần còn lại

Gặp `None` đầu tiên, phải kiểm tra **mọi** phần tử sau đó cũng là `None`.

In [22]:
t = counters["BFS không kiểm tra phần còn lại"]
print("cây", tree_to_list(t), "| sai ->", wrong_bfs_no_rest_check(t), "| đúng ->", Solution().isCompleteTree(t))
assert wrong_bfs_no_rest_check(t) != Solution().isCompleteTree(t)

cây [1, None, 2] | sai -> True | đúng -> False


Vì sao sai: sau `None` có thể còn nút thật (ví dụ `7` ở ví dụ 2). **Đúng:** đặt cờ `seen_none` và trả `False` nếu thấy nút thật sau đó.

### Lỗi 3 — Chỉ kiểm tra "có con phải thì phải có con trái"

Điều kiện này **cần** nhưng **không đủ**: tầng cuối còn phải dồn trái và các tầng trên phải đầy.

In [23]:
t = counters["chỉ kiểm tra con phải cần con trái"]
print("cây", tree_to_list(t), "| sai ->", wrong_right_needs_left(t), "| đúng ->", Solution().isCompleteTree(t))
assert wrong_right_needs_left(t) != Solution().isCompleteTree(t)

cây [1, 2, None, 3] | sai -> True | đúng -> False


Vì sao sai: ví dụ `[1,2,3,None,None,4,5]` (nút `2` không có con nhưng tầng 3 đã có nút ở bên phải). **Đúng:** xét thứ tự theo tầng.

### Lỗi 4 — Chỉ số 0-based sai

Với gốc chỉ số `0`, con của `i` phải là `2i + 1` và `2i + 2` (không phải `2i`, `2i + 1`); hoặc dùng 1-based với `2i`, `2i + 1`.

In [24]:
t = counters["chỉ số 0-based sai"]
print("cây", tree_to_list(t), "| sai ->", wrong_index_zero_based(t), "| đúng ->", Solution().isCompleteTree(t))
assert wrong_index_zero_based(t) != Solution().isCompleteTree(t)

cây [1, None, 2] | sai -> True | đúng -> False


Vì sao sai: với gốc `0`, `2 * 0 = 0` nên con trái trùng chỉ số với gốc. **Đúng:** 1-based (`2i`, `2i + 1`) hoặc 0-based (`2i + 1`, `2i + 2`).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "kiểm tra cây hoàn chỉnh" ⇒ **BFS theo tầng, ghi cả con rỗng**; sau `None` đầu tiên không được còn nút thật.
- **Chỉ số đống** (gốc `1`, `2i`, `2i + 1`): hoàn chỉnh ⇔ chỉ số lớn nhất `= n`.
- **Phân biệt:** đầy (full: 0 hoặc 2 con), hoàn hảo (perfect: mọi tầng đầy), hoàn chỉnh (complete: tầng cuối dồn trái).
- Cùng họ: Count Complete Tree Nodes (222), Maximum Depth of Binary Tree (104), Serialize and Deserialize Binary Tree (297), Binary Tree Level Order Traversal (102).
- Test nên có: một nút, chỉ con trái, chỉ con phải, tầng cuối có lỗ ở giữa, cây đầy nhưng không hoàn chỉnh, cây hoàn hảo, `n = 100`.

## 15. Biến thể

1. **Đếm số nút của cây hoàn chỉnh** nhanh hơn `O(n)` (bài tập 5, Count Complete Tree Nodes 222): `O(log^2 n)` dựa vào chiều cao hai biên.
2. **Vị trí vi phạm đầu tiên** (chỉ số đống của nút thật đầu tiên đứng sau một chỗ trống).
3. **Dựng cây hoàn chỉnh** từ danh sách theo mức.
4. Bài liên quan: 222.

Dưới đây là biến thể 2 kiểm bằng oracle.

In [25]:
def first_violation_index(root):
    """Chỉ số đống của nút thật đầu tiên xuất hiện sau một vị trí trống (theo thứ tự tầng), hoặc None nếu hoàn chỉnh."""
    queue = deque([(root, 1)])
    seen_none = False
    while queue:
        node, i = queue.popleft()
        if node is None:
            seen_none = True
            continue
        if seen_none:
            return i
        queue.append((node.left, 2 * i))
        queue.append((node.right, 2 * i + 1))
    return None

for n in range(1, 8):
    for sh in all_shapes(n):
        t = tree_from_shape(sh)
        v = first_violation_index(t)
        assert (v is None) == is_complete_shape(t), (n, sh)
        if v is not None:
            idx = indices(t)
            assert v in idx and any(j not in idx for j in range(1, v))   # nút thật đứng sau một chỉ số bị thiếu
print("ví dụ 2: nút vi phạm đầu tiên ở chỉ số", first_violation_index(build_tree([1, 2, 3, 4, 5, None, 7])))

ví dụ 2: nút vi phạm đầu tiên ở chỉ số 7


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So với cây chuẩn | Hình dạng chuẩn từ chỉ số đống | `O(n)` | `O(n)` | Oracle theo cấu trúc |
| 2. BFS + `None` | Sau `None` không còn nút thật | `O(n)` | `O(n)` | **Cách nộp chính** |
| 3. Chỉ số đống | `max_index == n` | `O(n)` | `O(n)` | Gọn; chú ý tràn ở ngôn ngữ khác |
| 4. Đệ quy bộ ba | `(complete, perfect, height)` | `O(n)` | `O(h)` | Không cần hàng đợi |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
t = build_tree([1, 2, 3, 4, 5, 6])
print(tree_to_list(t), count_nodes(t))
print(level_sequence(build_tree([1, 2, 3]))[:7])
print(complete_shape(3))
```

In [26]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với các danh sách theo mức sau, ghi `True/False` (cây hoàn chỉnh hay không): `[1,2,3,4,5,6]`, `[1,2,3,4,5,None,7]`, `[1,2,None,3]`, `[1,2,3,None,4]`.

In [27]:
# Bài 2: điền kết quả tính tay
hand_answers = None           # danh sách 4 giá trị bool theo thứ tự trên

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành BFS có nút rỗng.

In [28]:
def exercise_3_bfs(root: Optional[TreeNode]) -> bool:
    queue = deque([root])
    seen_none = False
    while queue:
        node = queue.popleft()
        # TODO 1: nếu node là None thì đặt seen_none = True và continue
        # TODO 2: nếu đã seen_none (mà node thật) thì trả False
        # TODO 3: đẩy node.left và node.right (kể cả None) vào hàng đợi
        pass
    return True


# for vals, expected in examples:
#     assert exercise_3_bfs(build_tree(vals)) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_index(root)` dùng chỉ số đống 1-based.

In [29]:
def exercise_4_index(root: Optional[TreeNode]) -> bool:
    # TODO
    pass


# for vals, expected in examples:
#     assert exercise_4_index(build_tree(vals)) == expected

### Bài 5 — Mini challenge: đếm nút của cây hoàn chỉnh nhanh (khó)

Viết `exercise_5_count(root)` đếm số nút của một cây **đã biết là hoàn chỉnh** với thời gian `O(log^2 n)`: đo chiều cao đi theo nhánh trái và nhánh phải; nếu bằng nhau thì cây con là hoàn hảo (`2^h - 1` nút), ngược lại đệ quy hai cây con.

```python
exercise_5_count(complete_tree(100)) -> 100
exercise_5_count(complete_tree(1))   -> 1
exercise_5_count(None)               -> 0
```

In [30]:
def exercise_5_count(root: Optional[TreeNode]) -> int:
    # TODO
    pass


# for n in (0, 1, 2, 7, 8, 63, 100):
#     assert exercise_5_count(complete_tree(n) if n else None) == n

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `tree_to_list` bỏ `None` thừa ở cuối; `complete_shape(3)` là hình dạng cây hoàn chỉnh 3 nút.
- Mức 2: `level_sequence` ghi cả `None`: sau `1, 2, 3` là bốn `None`.
- Mức 3: `[1, 2, 3, 4, 5, 6] 6`, `[1, 2, 3, None, None, None, None]`, `((None, None), (None, None))`.

**Bài 2**
- Mức 1: tìm `None` rồi xem có nút thật sau nó theo thứ tự tầng không.
- Mức 2: `[1,2,3,4,5,None,7]`: sau vị trí trống của con trái của `3` có `7`.
- Mức 3: `[True, False, False, False]`.

**Bài 3**
- Mức 1: `if node is None: seen_none = True; continue`.
- Mức 2: `if seen_none: return False`.
- Mức 3: `queue.append(node.left); queue.append(node.right)`.

**Bài 4**
- Mức 1: duyệt với cặp `(nút, chỉ số)`, gốc chỉ số `1`.
- Mức 2: con trái `2i`, con phải `2i + 1`; đếm số nút và lấy chỉ số lớn nhất.
- Mức 3: `return max_index == count`.

**Bài 5**
- Mức 1: hàm `depth(node, side)` đi dọc nhánh trái hoặc phải.
- Mức 2: nếu `depth_left == depth_right` thì `2**h - 1` nút.
- Mức 3: ngược lại `1 + đếm(trái) + đếm(phải)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [31]:
t = build_tree([1, 2, 3, 4, 5, 6])
print(tree_to_list(t), count_nodes(t))
print(level_sequence(build_tree([1, 2, 3]))[:7])
print(complete_shape(3))

assert (tree_to_list(t), count_nodes(t)) == ([1, 2, 3, 4, 5, 6], 6)
assert level_sequence(build_tree([1, 2, 3]))[:7] == [1, 2, 3, None, None, None, None]
assert complete_shape(3) == ((None, None), (None, None))

[1, 2, 3, 4, 5, 6] 6
[1, 2, 3, None, None, None, None]
((None, None), (None, None))


### Đáp án Bài 2

`[True, False, False, False]`.

In [32]:
hand_answers_solution = [True, False, False, False]
lists = [[1, 2, 3, 4, 5, 6], [1, 2, 3, 4, 5, None, 7], [1, 2, None, 3], [1, 2, 3, None, 4]]
assert [Solution().isCompleteTree(build_tree(v)) for v in lists] == hand_answers_solution
assert [is_complete_shape(build_tree(v)) for v in lists] == hand_answers_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [33]:
def exercise_3_bfs_solution(root: Optional[TreeNode]) -> bool:
    queue = deque([root])
    seen_none = False
    while queue:
        node = queue.popleft()
        if node is None:
            seen_none = True
            continue
        if seen_none:
            return False
        queue.append(node.left)
        queue.append(node.right)
    return True

for vals, expected in examples:
    assert exercise_3_bfs_solution(build_tree(vals)) == expected
for n in range(1, 8):
    for sh in all_shapes(n):
        t = tree_from_shape(sh)
        assert exercise_3_bfs_solution(t) == is_complete_shape(t)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [34]:
def exercise_4_index_solution(root: Optional[TreeNode]) -> bool:
    count, max_index = 0, 0
    queue = deque([(root, 1)])
    while queue:
        node, i = queue.popleft()
        if node is None:
            continue
        count += 1
        max_index = max(max_index, i)
        queue.append((node.left, 2 * i))
        queue.append((node.right, 2 * i + 1))
    return max_index == count

for vals, expected in examples:
    assert exercise_4_index_solution(build_tree(vals)) == expected
for n in range(1, 8):
    for sh in all_shapes(n):
        t = tree_from_shape(sh)
        assert exercise_4_index_solution(t) == is_complete_shape(t)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [35]:
def exercise_5_count_solution(root: Optional[TreeNode]) -> int:
    def depth(node, side):
        d = 0
        while node:
            d += 1
            node = node.left if side == "L" else node.right
        return d

    if root is None:
        return 0
    hl, hr = depth(root, "L"), depth(root, "R")
    if hl == hr:
        return (1 << hl) - 1
    return 1 + exercise_5_count_solution(root.left) + exercise_5_count_solution(root.right)

assert exercise_5_count_solution(None) == 0
assert exercise_5_count_solution(complete_tree(1)) == 1
for n in range(1, 101):
    assert exercise_5_count_solution(complete_tree(n)) == n, n
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (BFS ghi cả con rỗng)

```python
class Solution:
    def isCompleteTree(self, root: Optional[TreeNode]) -> bool:
        from collections import deque
        queue = deque([root])
        seen_none = False                    # đã gặp một vị trí trống chưa
        while queue:
            node = queue.popleft()
            if node is None:
                seen_none = True
                continue
            if seen_none:                    # nút thật xuất hiện sau chỗ trống -> không hoàn chỉnh
                return False
            queue.append(node.left)          # đẩy cả None
            queue.append(node.right)
        return True
```

Công thức nhớ nhanh: theo tầng, sau `None` đầu tiên không còn nút thật; hoặc chỉ số đống lớn nhất `= n`.

### Checklist trước khi submit

- [ ] Đẩy **cả con rỗng** (`None`) vào hàng đợi?
- [ ] Sau nút rỗng đầu tiên, gặp nút thật ⇒ `False`?
- [ ] Không nhầm với cây đầy (full) hay cây hoàn hảo (perfect)?
- [ ] Nếu dùng chỉ số đống: 1-based với `2i`, `2i + 1` (hoặc 0-based `2i + 1`, `2i + 2`)?
- [ ] Đã thử: một nút, chỉ con phải, tầng cuối có lỗ, cây đầy không hoàn chỉnh, `n = 100`?
- [ ] Tên method đúng `isCompleteTree(self, root)` và trả `bool`?